# Sizing the bias_gm Core

Adapted from `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`, which is the overview of the pygmid lookup commands this notebook uses.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

bias_gm was sized by hand equations and simulation sweeps, so this notebook rebuilds the derivation after the fact. It starts from the spec (200 nA at code 4, V_R of at least 100 mV, phase margin of about 45 deg, supply 1.45 to 1.55 V, OTA bias pin 0.35 to 0.8 V), derives the sizes with gm/ID lookups, then reads the drawn sizes from the schematic and compares. Nothing here changes the schematic. Where the derivation disagrees with the drawing, the cell says so and leaves the drawing alone.

Every number below is a gm/ID model estimate from the tt tables at 300 K, with no mismatch, no process corner and no layout wiring unless a cell says otherwise. The simulated values, the phase margin among them, are in `macros/bias_gm/README.md`.

The core is the Pavan supply-rejection loop with a mixed-threshold pair. `M2` is an LV NMOS diode on `R1`, `M1` an HV NMOS on the same gate `vx`, both fed with the same current by the PMOS mirror `M3` and `M4`. The loop sense copy `M6` drives the PMOS diode `M5` and holds `vy` at `vx`. The output is the trimmable mirror `bias_trim` on `vz`.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

In [2]:
# Spec and the choices the README gives
VDD_MIN, VDD, VDD_MAX = 1.45, 1.5, 1.55
I_OUT   = 200e-9      # A, output at code 4
CODE    = 4
N_OUT   = 16 + 2 * CODE   # output units at that code
N_CORE  = 24          # PMOS units per core leg, fixed by 200 nA / 8.33 nA
VR_MIN  = 0.100       # V, hard rule on I*R1
PM_MIN  = 45          # deg, loop phase margin target
V_PIN   = (0.35, 0.8) # V, OTA bias pin range
I_BR    = 100e-9      # A, branch M5 / M6, see the branch cell
UNIT_N  = (0.5, 10.0) # um, W and L of the NMOS unit
UNIT_P  = (0.5, 5.0)  # um, W and L of the PMOS unit
K_UT    = 1.380649e-23 * 300 / 1.602177e-19   # thermal voltage at the table temperature
CELL    = 'bias_gm'
DATA    = Path('data')
SCH     = Path('../schematic') / (CELL + '.sch')

### Read the sizes out of the schematic

The schematic is the source of truth, so nothing here restates a drawn size. `read_sizes` returns W, L (um) and m per device name. The checks below are the invariants the topology rests on: K = 1 means `M1` and `M2` carry the same total W/L, the core mirror legs and the diode share one unit device, and the PMOS unit equals the one in `bias_trim`.

In [3]:
def num(s):
    # width and length come as 0.5u or as 0.5e-6 on the rhigh symbol
    return float(s[:-1]) if s.endswith('u') else float(s) * 1e6

def read_sizes(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'^C \{([^}]+)\.sym\} \S+ \S+ \S+ \S+ \{name=(\S+)\n(.*?)\}', txt, re.S | re.M):
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(3), re.M))
        if 'w' in a and 'l' in a:
            dev[m.group(2)] = dict(w=num(a['w']), l=num(a['l']), m=int(a.get('m', 1)), sym=m.group(1))
    return dev, txt

D, SCH_TXT = read_sizes(SCH)
tot = lambda n: D[n]['w'] / D[n]['l'] * D[n]['m']
N_TRIM = len(re.findall(r'bias_trim\.sym', SCH_TXT))

for n in ('M1', 'M2', 'M6', 'MSN', 'M3', 'M4', 'M5', 'CC', 'R1', 'RZ'):
    d = D[n]
    print(f"{n:4s} {d['sym']:14s} W {d['w']:7.2f} um  L {d['l']:7.2f} um  m {d['m']:3d}")
print(f'bias_trim instances on vz: {N_TRIM}')

assert (D['M1']['w'], D['M1']['l']) == UNIT_N == (D['M2']['w'], D['M2']['l']) == (D['M6']['w'], D['M6']['l'])
assert abs(tot('M1') - tot('M2')) < 1e-12, 'K is not 1'
assert (D['M3']['w'], D['M3']['l']) == UNIT_P == (D['M4']['w'], D['M4']['l']) == (D['M5']['w'], D['M5']['l'])
assert D['M3']['m'] == D['M4']['m'] == N_CORE

M1   sg13_hv_nmos   W    0.50 um  L   10.00 um  m   4
M2   sg13_lv_nmos   W    0.50 um  L   10.00 um  m   4
M6   sg13_hv_nmos   W    0.50 um  L   10.00 um  m   2
MSN  sg13_hv_nmos   W    0.50 um  L   10.00 um  m   2
M3   sg13_lv_pmos   W    0.50 um  L    5.00 um  m  24
M4   sg13_lv_pmos   W    0.50 um  L    5.00 um  m  24
M5   sg13_lv_pmos   W    0.50 um  L    5.00 um  m  12
CC   sg13_hv_nmos   W   10.00 um  L   10.00 um  m  20
R1   rhigh          W    0.50 um  L  735.00 um  m   1
RZ   rhigh          W    0.50 um  L   43.90 um  m   1
bias_trim instances on vz: 2


In [4]:
# Lookups: HV and LV NMOS and LV PMOS, extensive quantities are per um of width
tabs = {n: lk(str(DATA / f'sg13cmos5l_{n}.mat')) for n in ('hv_nmos', 'lv_nmos', 'lv_pmos')}
hn, ln, lp = tabs['hv_nmos'], tabs['lv_nmos'], tabs['lv_pmos']

def per_um(t, f, L, vgs, vds, vsb=0.0):
    # GM, GDS, CGG, CDD, ID ... of one um of width
    return float(t.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb)) / float(t['W'])

def ratio(t, f, L, vgs, vds, vsb=0.0):
    return float(t.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb))

def vgs_for(t, idw, L, vds, vsb=0.0):
    return float(t.lookupVGS(ID_W=idw, L=L, VDS=vds, VSB=vsb))

def n_slope(t, L, vds, vsb=0.0):
    # weak inversion plateau of gm/ID is 1/(n UT)
    vg = np.arange(0.0, 0.9, 0.01)
    return 1 / (K_UT * np.max(t.lookup('GM_ID', L=L, VGS=vg, VDS=vds, VSB=vsb)))

def ic_from(gm_id, n):
    # EKV: gm/ID * n UT = 2 / (1 + sqrt(1 + 4 IC))
    return max(((2 / (gm_id * n * K_UT) - 1) ** 2 - 1) / 4, 0.0)

### The resistor voltage

With K = 1 the equal currents in `M1` and `M2` give I R1 = V_GS1 - V_GS2, and the two gate-source voltages differ by the HV against LV threshold at the same current density. `M2` has its source on `R1`, so it sees V_SB = V_R and a body effect, and `M1` sits at V_DS = vx. The loop makes `vy` equal to `vx`, so one fixed point solves both.

The current per unit is the core leg over the unit count of `M1`. The unit count is drawn, the cell after next shows why it is four.

In [5]:
N1 = D['M1']['m']
I_UNIT_N = I_OUT * N_CORE / N_OUT / N1       # core leg is 24 units of I_OUT/N_OUT
I_CORE   = I_OUT * N_CORE / N_OUT
IDW_N    = I_UNIT_N / UNIT_N[0]               # A/um
L = UNIT_N[1]

vr = 0.4
for _ in range(50):
    vgs1 = vgs_for(hn, IDW_N, L, vds=vgs_for(hn, IDW_N, L, 0.7), vsb=0)   # vds = vx = vgs1, one inner pass
    vgs2 = vgs_for(ln, IDW_N, L, vds=0.3, vsb=vr)
    vgs2 = vgs_for(ln, IDW_N, L, vds=vgs2, vsb=vr)
    vr_new = vgs1 - vgs2
    if abs(vr_new - vr) < 1e-6:
        break
    vr = vr_new
VR = vr
vth1 = ratio(hn, 'VTH', L, vgs1, vgs1, 0)
vth2 = ratio(ln, 'VTH', L, vgs2, vgs2, VR)
print(f'core current {I_CORE*1e9:.0f} nA, {I_UNIT_N*1e9:.1f} nA per NMOS unit, {IDW_N*1e9:.0f} nA/um')
print(f'VGS1 (HV)  {vgs1*1e3:6.1f} mV   VTH {vth1*1e3:6.1f} mV   at VDS = {vgs1:.3f} V')
print(f'VGS2 (LV)  {vgs2*1e3:6.1f} mV   VTH {vth2*1e3:6.1f} mV   at VSB = {VR:.3f} V')
print(f'V_R = VGS1 - VGS2 = {VR*1e3:.1f} mV, dVth at the operating points {1e3*(vth1-vth2):.1f} mV, K = 1 adds n UT ln K = 0')
print(f'overdrive VGS - VTH: HV {1e3*(vgs1-vth1):.1f} mV, LV {1e3*(vgs2-vth2):.1f} mV, so the pair adds {1e3*((vgs1-vth1)-(vgs2-vth2)):.1f} mV on top of the threshold difference')
print(f'hard rule {VR_MIN*1e3:.0f} mV: margin {VR*1e3 - VR_MIN*1e3:.0f} mV, sim says 421 mV at tt')

core current 200 nA, 50.0 nA per NMOS unit, 100 nA/um
VGS1 (HV)   704.0 mV   VTH  558.4 mV   at VDS = 0.704 V
VGS2 (LV)   285.5 mV   VTH  225.5 mV   at VSB = 0.419 V
V_R = VGS1 - VGS2 = 418.5 mV, dVth at the operating points 332.9 mV, K = 1 adds n UT ln K = 0
overdrive VGS - VTH: HV 145.6 mV, LV 60.0 mV, so the pair adds 85.6 mV on top of the threshold difference
hard rule 100 mV: margin 319 mV, sim says 421 mV at tt


The README writes V_R = dVth + n UT ln K. With K = 1 the second term is not zero in this table, because the HV unit runs at a higher inversion level than the LV unit at the same current (the next cells give 4.3 and 1.3). The 85 mV of overdrive difference is the model's version of that term, and the sum lands on the simulated 421 mV within 3 mV.

Why the mixed pair at all. With two LV devices the resistor voltage is only n UT ln K, so reaching 100 mV with K as the only knob costs a ratio of about 26 (table below) and a matching penalty on top. With the HV/LV pair V_R comes from the oxide difference and K can stay 1.

In [6]:
n_lv = n_slope(ln, L, 0.3, VR)
print(f'n of the LV unit from the gm/ID plateau: {n_lv:.2f}')
for k in (1, 2, 4, 8, 16, 20, 24):
    print(f'LV pair, K = {k:2d}: V_R = {n_lv*K_UT*np.log(k)*1e3:5.1f} mV')
print(f'K needed for {VR_MIN*1e3:.0f} mV: {np.exp(VR_MIN/(n_lv*K_UT)):.1f}')

n of the LV unit from the gm/ID plateau: 1.18
LV pair, K =  1: V_R =   0.0 mV
LV pair, K =  2: V_R =  21.2 mV
LV pair, K =  4: V_R =  42.4 mV
LV pair, K =  8: V_R =  63.6 mV
LV pair, K = 16: V_R =  84.8 mV
LV pair, K = 20: V_R =  91.6 mV
LV pair, K = 24: V_R =  97.2 mV
K needed for 100 mV: 26.3


### R1

The resistor is the voltage over the core current. The sheet value is not in the lookup tables, so it comes from the README (2.17 MOhm for 0.5u/735u at tt 27 C, 1470 squares). It is the one input in this notebook that is not a table value.

In [7]:
RSQ = 2.17e6 / (735 / 0.5)                   # ohm/sq, README datum
R1_REQ = VR / I_CORE
L_REQ = R1_REQ / RSQ * 0.5
print(f'R1 = V_R / I = {R1_REQ/1e6:.3f} MOhm -> 0.5u/{L_REQ:.0f}u')
print(f"drawn: 0.5u/{D['R1']['l']:.0f}u = {D['R1']['l']/0.5*RSQ/1e6:.3f} MOhm, {100*(D['R1']['l']/L_REQ-1):+.1f} % against the derivation")
print('README: 713.5u centred the sim core at 200 nA, 735u is 3 % longer on purpose (joint power limit)')

R1 = V_R / I = 2.093 MOhm -> 0.5u/709u
drawn: 0.5u/735u = 2.170 MOhm, +3.7 % against the derivation
README: 713.5u centred the sim core at 200 nA, 735u is 3 % longer on purpose (joint power limit)


### Inversion level of the 0.5u/10u units

The unit is set by two limits. The HV device is at its maximum length of 10 um, and the current per unit has to be small enough that gm1 stays low, since gm1 and the gate area on `vx` set the `vx` pole. The README targets I/Is of about 4. Inversion coefficient from the table gm/ID and the plateau slope n, for both devices at the drawn current.

In [8]:
n_hv = n_slope(hn, L, vgs1)
gm_hv = ratio(hn, 'GM_ID', L, vgs1, vgs1, 0)
gm_lv = ratio(ln, 'GM_ID', L, vgs2, vgs2, VR)
ic_hv, ic_lv = ic_from(gm_hv, n_hv), ic_from(gm_lv, n_lv)
print(f'HV unit: n {n_hv:.2f}, gm/ID {gm_hv:5.2f} 1/V, I/Is {ic_hv:.2f}')
print(f'LV unit: n {n_lv:.2f}, gm/ID {gm_lv:5.2f} 1/V, I/Is {ic_lv:.2f}')
print(f'drawn total W/L of M1: {tot("M1"):.2f}, M2: {tot("M2"):.2f}, M6: {tot("M6"):.2f}, MSN: {tot("MSN"):.2f}')

# unit count for an I/Is target, per device, at the same current
IC_T = 4.0
def units_for_ic(t, n, vds, vsb):
    vg = np.arange(0.05, 1.2, 0.002)
    gmid = t.lookup('GM_ID', L=L, VGS=vg, VDS=vds, VSB=vsb)
    ic = np.array([ic_from(g, n) for g in gmid])
    k = int(np.argmin(abs(ic - IC_T)))
    idw = per_um(t, 'ID', L, vg[k], vds, vsb) 
    return I_CORE / (idw * UNIT_N[0]), vg[k]
for nm, t, n, vds, vsb in (('HV', hn, n_hv, vgs1, 0.0), ('LV', ln, n_lv, vgs2, VR)):
    nu, vg = units_for_ic(t, n, vds, vsb)
    print(f'{nm} at I/Is = {IC_T:.0f}: {nu:.1f} units of 0.5u/10u at 200 nA, VGS {vg*1e3:.0f} mV')

HV unit: n 1.37, gm/ID 10.66 1/V, I/Is 4.33
LV unit: n 1.18, gm/ID 18.76 1/V, I/Is 1.30
drawn total W/L of M1: 0.20, M2: 0.20, M6: 0.10, MSN: 0.10
HV at I/Is = 4: 4.3 units of 0.5u/10u at 200 nA, VGS 696 mV
LV at I/Is = 4: 1.3 units of 0.5u/10u at 200 nA, VGS 360 mV


Both devices carry the same total W/L for K = 1. The HV device sets gm1 and the `vx` loading, so the I/Is target applies to it: 4.3 units, drawn 4 (I/Is 4.3). The LV device at the same W/L runs at I/Is 1.3, which is fine because V_R comes from the oxide difference and not from the LV inversion level.

Why L = 10 um. The table below holds V_R and gm1 for the L the tables offer, at the same current density.

In [9]:
print('  L/um   VGS_HV/mV  VGS_LV/mV   V_R/mV   gm1/uS (4 units)')
for l in (0.5, 1, 5, 10):
    vh = vgs_for(hn, IDW_N, l, 0.7)
    vh = vgs_for(hn, IDW_N, l, vh)
    v = 0.4
    for _ in range(30):
        vl = vgs_for(ln, IDW_N, l, 0.3, v); vl = vgs_for(ln, IDW_N, l, vl, v); v = vh - vl
    gm1 = per_um(hn, 'GM', l, vh, vh) * UNIT_N[0] * N1
    print(f'{l:6.1f}  {vh*1e3:9.1f}  {vl*1e3:9.1f}  {v*1e3:7.1f}  {gm1*1e6:8.3f}')

  L/um   VGS_HV/mV  VGS_LV/mV   V_R/mV   gm1/uS (4 units)


   0.5      584.5      239.8    344.7     5.363


   1.0      602.1      235.5    366.6     4.556


   5.0      651.4      261.1    390.3     2.728
  10.0      704.0      285.5    418.5     2.131


From 0.5 um to 10 um V_R rises from 345 to 419 mV and gm1 falls from 5.4 to 2.1 uS, both in the direction the loop wants (more V_R, a smaller gm1 on the `vx` pole). L = 10 um is also the longest single device the PDK symbol accepts.

### PMOS mirror unit 0.5u/5u

The unit current follows from the 24 units per leg: 200 nA over 24 is 8.33 nA. The saturation margin is checked at the two places that matter. The core legs `M3` and `M4` have their drains at `vy` and `vx`, about 0.7 V. The output units of `bias_trim` see the OTA bias pin, 0.35 to 0.8 V, through the drain switch.

In [10]:
I_UNIT_P = I_OUT / N_OUT
IDW_P = I_UNIT_P / UNIT_P[0]
LP = UNIT_P[1]
vx = vgs1                                    # vx = VGS1 with vy = vx
print(f'PMOS unit {I_UNIT_P*1e9:.2f} nA, {IDW_P*1e9:.1f} nA/um at L = {LP} um')
print('place                       VSD/V   VSG/mV  gm/ID  VDSS/mV  margin/mV')
for nm, vsd in (('core leg, vy = vx', VDD - vx),
                ('pin 0.35 V, VDD 1.55', 1.55 - V_PIN[0]),
                ('pin 0.80 V, VDD 1.45', VDD_MIN - V_PIN[1])):
    vsg = vgs_for(lp, IDW_P, LP, vsd)
    print(f'{nm:26s} {vsd:6.3f}  {vsg*1e3:7.1f}  {ratio(lp,"GM_ID",LP,vsg,vsd):5.2f}  {ratio(lp,"VDSS",LP,vsg,vsd)*1e3:7.1f}  {(vsd-ratio(lp,"VDSS",LP,vsg,vsd))*1e3:8.0f}')
VSG_P = vgs_for(lp, IDW_P, LP, VDD - vx)
print(f'vz = VDD - VSG = {1e3*(VDD-VSG_P):.0f} mV at VDD 1.5 V, so vz follows VDD and the units keep the same VSG')

PMOS unit 8.33 nA, 16.7 nA/um at L = 5.0 um
place                       VSD/V   VSG/mV  gm/ID  VDSS/mV  margin/mV
core leg, vy = vx           0.796    340.4  24.55    110.8       685


pin 0.35 V, VDD 1.55        1.200    340.2  24.61    110.7      1089
pin 0.80 V, VDD 1.45        0.650    340.5  24.52    110.8       539
vz = VDD - VSG = 1160 mV at VDD 1.5 V, so vz follows VDD and the units keep the same VSG


The unit sits in weak inversion (gm/ID about 25 1/V, VSG 340 mV) with a saturation voltage of 111 mV. The worst pin point (0.8 V at VDD 1.45 V) leaves 0.65 V of VSD, 539 mV above VDSS before the 15 mV of the trim switch. Saturation is not the constraint on the PMOS side, output resistance is, which is why L = 5 um.

Why L = 5 um. The units at W = 0.5 um and the same 8.33 nA, over the lengths the grid offers. `rout` is for the 24 units of the output at code 4 at the worst pin voltage, and the span is the current change over the 0.35 to 0.8 V pin range from that resistance alone.

In [11]:
VSD_W = VDD_MIN - V_PIN[1]
print('  L/um  VSG/mV  gm/ID  gm/gds  rout24/GOhm  span 0.45 V/%   gate area/um2   Cgg/fF')
rows = {}
for l in (0.5, 1, 5, 10):
    vsg = vgs_for(lp, IDW_P, l, VSD_W)
    gm = per_um(lp, 'GM', l, vsg, VSD_W) * UNIT_P[0]
    gds = per_um(lp, 'GDS', l, vsg, VSD_W) * UNIT_P[0]
    cgg = per_um(lp, 'CGG', l, vsg, VSD_W) * UNIT_P[0]
    r24 = 1 / (N_OUT * gds)
    span = 100 * (V_PIN[1] - V_PIN[0]) / r24 / I_OUT
    rows[l] = (gm, gds, cgg)
    print(f'{l:6.1f} {vsg*1e3:7.1f} {gm/I_UNIT_P:6.2f} {gm/gds:7.0f} {r24/1e9:11.3f} {span:13.3f} {UNIT_P[0]*l:14.2f} {cgg*1e15:8.2f}')

  L/um  VSG/mV  gm/ID  gm/gds  rout24/GOhm  span 0.45 V/%   gate area/um2   Cgg/fF
   0.5   277.4  31.03     582       0.094         2.401           0.25     1.04
   1.0   290.6  31.44    1024       0.163         1.381           0.50     1.84
   5.0   340.5  24.99    1310       0.262         0.859           2.50    11.66
  10.0   370.4  20.93    1467       0.350         0.642           5.00    27.86


The table span is 0.86 % at L = 5 um against 1.4 % at 1 um and 2.4 % at 0.5 um. L = 10 um gives 0.64 %, but doubles the gate area (5.0 against 2.5 um2) and takes 2.4 times the gate capacitance per unit (27.9 against 11.7 fF), which lands on `vz` 152 times and moves its pole. The simulated span of `bias_trim` is 0.88 % at the hot corner (README), so the table value is about what the cell does. L = 5 um is the step that meets the 1 % span without paying the capacitance.

### Loop gain

A small-signal model of the three core nodes `vx`, `vy`, `vz` and the source node of `M2`, with the loop opened at the gate of `M6` as `g5.py` does (T = -v(vy) / v(g6)). Every conductance comes from the table at the operating point found above. `M2` carries the body effect through GMB.

In [12]:
Wu, Lu = UNIT_N; Wp, Lp = UNIT_P
def nm_par(t, l, vgs, vds, vsb, n_units, w=Wu):
    g = lambda f: per_um(t, f, l, vgs, vds, vsb) * w * n_units
    return dict(gm=g('GM'), gds=g('GDS'), gmb=g('GMB'), cgg=g('CGG'), cdd=g('CDD'))

vsd_core = VDD - vx
m1 = nm_par(hn, L, vgs1, vgs1, 0, D['M1']['m'])
m2 = nm_par(ln, L, vgs2, vgs2, VR, D['M2']['m'])
m6 = nm_par(hn, L, vgs1, vx, 0, D['M6']['m'])
vsg_c = vgs_for(lp, IDW_P, LP, vsd_core)
p = lambda n_units: nm_par(lp, LP, vsg_c, vsd_core, 0, n_units, Wp)
m3, m4, m5 = p(D['M3']['m']), p(D['M4']['m']), p(D['M5']['m'])
for k, v in (('M1', m1), ('M2', m2), ('M3', m3), ('M4', m4), ('M5', m5), ('M6', m6)):
    print(f"{k}: gm {v['gm']*1e6:7.3f} uS  gds {v['gds']*1e9:7.3f} nS  gmb {v['gmb']*1e6:6.3f} uS  gm/gds {v['gm']/v['gds']:6.0f}")
print(f"gm4 R1 = {m4['gm']*D['R1']['l']/0.5*RSQ:5.2f}   (V_R x gm/ID of the PMOS unit {VR*m4['gm']/I_CORE:5.2f}), README says about 13")
print('M2 gm/gds is low in the table (LV, L = 10 um, weak inversion), which does not matter in a diode')

M1: gm   2.131 uS  gds   3.135 nS  gmb  0.696 uS  gm/gds    680
M2: gm   3.740 uS  gds  88.835 nS  gmb  0.404 uS  gm/gds     42
M3: gm   5.006 uS  gds   3.093 nS  gmb  0.943 uS  gm/gds   1618
M4: gm   5.006 uS  gds   3.093 nS  gmb  0.943 uS  gm/gds   1618
M5: gm   2.503 uS  gds   1.547 nS  gmb  0.471 uS  gm/gds   1618
M6: gm   1.065 uS  gds   1.568 nS  gmb  0.348 uS  gm/gds    680
gm4 R1 = 10.86   (V_R x gm/ID of the PMOS unit 10.48), README says about 13
M2 gm/gds is low in the table (LV, L = 10 um, weak inversion), which does not matter in a diode


In [13]:
R1v = D['R1']['l'] / 0.5 * RSQ
RZv = D['RZ']['l'] / 0.5 * RSQ

def loop(s, cc_units=D['CC']['m'], c=None, rz=RZv):
    # unknowns vx, vs, vy, vc, vz, gate of M6 driven with 1 V, T = -vy / vg6
    c = c or CAPS
    cc = c['cc_per_unit'] * cc_units
    A = np.zeros((5, 5), complex); b = np.zeros(5, complex)
    g2 = m2['gm'] + m2['gds']
    # vx: s Cx vx + Id2 + gm4 vz + gds4 vx = 0, Id2 = g2 (vx - vs) - gmb2 vs
    A[0] = [s * c['vx'] + g2 + m4['gds'], -(g2 + m2['gmb']), 0, 0, m4['gm']]
    # vs: vs / R1 = Id2
    A[1] = [-g2, 1 / R1v + g2 + m2['gmb'], 0, 0, 0]
    # vy: s Cy vy + (vy - vc) / RZ + gm1 vx + gds1 vy + gm3 vz + gds3 vy = 0
    A[2] = [m1['gm'], 0, s * c['vy'] + 1 / rz + m1['gds'] + m3['gds'], -1 / rz, m3['gm']]
    # vc: (vc - vy) / RZ + s CC vc = 0
    A[3] = [0, 0, -1 / rz, 1 / rz + s * cc, 0]
    # vz: s Cz vz + (gm5 + gds5) vz + gm6 vg6 + gds6 vz = 0
    A[4] = [0, 0, 0, 0, s * c['vz'] + m5['gm'] + m5['gds'] + m6['gds']]
    b[4] = -m6['gm']
    v = np.linalg.solve(A, b)
    return -v[2]

### Capacitances at `vx`, `vy`, `vz` and CC

The node capacitances come from the table CGG and CDD of the devices that hang on each node, at the operating point of the cell. `M2` is counted as a gate cap to ground, its source sits behind `R1`. The wiring capacitances of the layout (`vz` 218 fF, `vx` 51 fF, README PEX section) are kept apart, so the model can be read with and without them.

`CC` is the HV NMOS gate cap with drain, source and bulk on vss and the gate on the RZ side, evaluated at the gate voltage `vy` = `vx`.

In [14]:
cgg_hv = per_um(hn, 'CGG', L, vx, 0.0, 0.0)           # F/um at 10 um length, source and drain on vss
cc_unit = per_um(hn, 'CGG', D['CC']['l'], vx, 0.0, 0.0) * D['CC']['w']
cpu = per_um(lp, 'CGG', LP, vsg_c, vsd_core) * Wp     # PMOS unit gate cap
cdd_pu = per_um(lp, 'CDD', LP, vsg_c, vsd_core) * Wp

N_VZ_GATES = D['M3']['m'] + D['M4']['m'] + D['M5']['m'] + 46 * N_TRIM
CAPS = dict(
    cc_per_unit = cc_unit,
    vx = (D['M1']['m'] + D['MSN']['m']) * m1['cgg'] / D['M1']['m'] + m2['cgg'] + D['M4']['m'] * cdd_pu,
    vy = D['M1']['m'] * m1['cdd'] / D['M1']['m'] + D['M3']['m'] * cdd_pu,    # gate of M6 is the loop cut
    vz = N_VZ_GATES * cpu + D['M5']['m'] * cdd_pu + D['M6']['m'] * m6['cdd'] / D['M6']['m'],
)
WIRE = dict(vx=50.6e-15, vy=28.7e-15, vz=217.8e-15)   # README PEX, schematic has none
print(f"CC {D['CC']['m']} units: {cc_unit*1e12:.3f} pF each, {cc_unit*D['CC']['m']*1e12:.2f} pF, README 5.8 pF for 16 units")
print(f"{D['CC']['w']*D['CC']['l']*D['CC']['m']:.0f} um2 of gate, bias_trim banks on vz: {N_TRIM}, gates on vz: {N_VZ_GATES}")
for k in ('vx', 'vy', 'vz'):
    print(f'C{k}: {CAPS[k]*1e15:8.1f} fF   with wiring {1e15*(CAPS[k]+WIRE[k]):8.1f} fF')

CC 20 units: 0.364 pF each, 7.27 pF, README 5.8 pF for 16 units
2000 um2 of gate, bias_trim banks on vz: 2, gates on vz: 152
Cvx:    196.3 fF   with wiring    246.9 fF
Cvy:     10.1 fF   with wiring     38.8 fF
Cvz:   1775.4 fF   with wiring   1993.2 fF


### Poles and zero by hand

The README formulas. `CC` on `vy` with the output conductance there sets the dominant pole. The `vx` pole is R1 plus the diode 1/gm2 into the gate capacitance on `vx`. The `vz` pole is gm5 over the capacitance on `vz`. The zero from RZ and CC sits at 1/(2 pi RZ CC).

In [15]:
f2pi = lambda x: x / (2 * np.pi)
CCt = cc_unit * D['CC']['m']
g_vy = m1['gds'] + m3['gds'] + 0.0
T0 = abs(loop(1e-6))
p_dom = f2pi(g_vy / CCt)
p_x = f2pi(1 / ((R1v + 1 / m2['gm']) * CAPS['vx']))
p_x_w = f2pi(1 / ((R1v + 1 / m2['gm']) * (CAPS['vx'] + WIRE['vx'])))
p_z = f2pi(m5['gm'] / CAPS['vz'])
p_z_w = f2pi(m5['gm'] / (CAPS['vz'] + WIRE['vz']))
z_rz = f2pi(1 / (RZv * CCt))
print(f'T(0) from the small-signal model: {20*np.log10(T0):.1f} dB, sim 55.4 to 64.0 dB over 135 PVT')
print(f"RZ {RZv/1e3:.0f} kOhm (README 130 kOhm)")
print(f'dominant pole at vy, g/CC:  {p_dom/1e3:8.2f} kHz   (README about 1 kHz, the model T(0) x pole = {T0*p_dom/1e3:.0f} kHz is the unity gain)')
print(f'vx pole:                    {p_x/1e3:8.0f} kHz   with wiring {p_x_w/1e3:.0f} kHz   (README about 150 kHz)')
print(f'vz pole:                    {p_z/1e3:8.0f} kHz   with wiring {p_z_w/1e3:.0f} kHz   (README about 250 kHz)')
print(f'RZ zero:                    {z_rz/1e3:8.0f} kHz   ({f2pi(1/(RZv*cc_unit*16))/1e3:.0f} kHz with 16 units, README about 210 kHz for 5.8 pF)')

T(0) from the small-signal model: 64.0 dB, sim 55.4 to 64.0 dB over 135 PVT
RZ 130 kOhm (README 130 kOhm)
dominant pole at vy, g/CC:      0.14 kHz   (README about 1 kHz, the model T(0) x pole = 215 kHz is the unity gain)
vx pole:                         333 kHz   with wiring 264 kHz   (README about 150 kHz)
vz pole:                         224 kHz   with wiring 200 kHz   (README about 250 kHz)
RZ zero:                         169 kHz   (211 kHz with 16 units, README about 210 kHz for 5.8 pF)


The model puts the dominant pole at 0.14 kHz, the README text says about 1 kHz. The model unity gain (T(0) of 64 dB times the pole, 209 kHz in the next cell) and the sim unity gain of 240 kHz at tt agree, so the 0.14 kHz is consistent and the README figure is the loose one. The `vx` pole is 333 kHz in the model (264 kHz with wiring) against about 150 kHz in the README, a factor of 1.8 to 2.2 that the model does not explain (the README figure is a reading from the simulated loop). The `vz` pole is 224 kHz against about 250 kHz.

### Phase margin from the model, and what the README quotes

The same small-signal model gives T(jw) with the poles and the zero in place. This is a hand-model check of where the unity-gain frequency lands, not the phase margin of the cell. The phase margin is a simulation result: 57.5 deg at tt 27 C and 46.9 to 65.4 deg over 135 PVT on the schematic, 48.2 deg and 40.9 to 56.3 deg on the PEX view before the test bank, 46.9 deg minimum with the test bank and `CC` of 20 units (README, loop with `g5.py`).

In [16]:
def margins(**kw):
    f = np.logspace(1, 8, 4000)
    T = np.array([loop(2j * np.pi * x, **kw) for x in f])
    mag = np.abs(T)
    k = np.where((mag[:-1] > 1) & (mag[1:] <= 1))[0]
    if not len(k):
        return None
    i = k[0]
    ph = np.degrees(np.unwrap(np.angle(T)))
    return f[i], 180 + ph[i] - 360 * np.round((ph[0]) / 360)

def pm_report(tag, **kw):
    r = margins(**kw)
    print(tag, 'no crossing' if r is None else f'unity gain {r[0]/1e3:7.1f} kHz, phase margin {r[1]:6.1f} deg')

pm_report('schematic caps, CC as drawn      ')
CAPS_S = dict(CAPS)
CAPS = {k: (CAPS_S[k] + WIRE[k] if k in WIRE else CAPS_S[k]) for k in CAPS_S}
pm_report('with PEX wiring on vx, vy, vz     ')
CAPS = CAPS_S

schematic caps, CC as drawn       unity gain   209.0 kHz, phase margin   55.5 deg


with PEX wiring on vx, vy, vz      unity gain   187.4 kHz, phase margin   47.6 deg


### Sizing CC for the target

`CC` scales the unity-gain frequency, the RZ zero comes with it. Units of the HV 10u/10u cell against the model phase margin, without and with the wiring. The README fixes 20 units for 46.9 deg on the PEX view with the test bank, so the target for the drawn `CC` is that sim number and not the model. In the model 16 units reach 46 deg on the schematic and 38 deg with the wiring, 20 units give 55.5 and 47.6 deg (sim tt: 57.5 schematic, 51.3 PEX), so the model reproduces the choice of 20 units for a 45 deg target on the layout.

In [17]:
print(' CC units   CC/pF   PM model   PM model with wiring')
for u in (8, 12, 16, 20, 24, 32, 42):
    r = margins(cc_units=u)
    CAPS = {k: (CAPS_S[k] + WIRE[k] if k in WIRE else CAPS_S[k]) for k in CAPS_S}
    rw = margins(cc_units=u)
    CAPS = CAPS_S
    f = lambda r: '   none' if r is None else f'{r[1]:7.1f}'
    print(f'{u:8d} {cc_unit*u*1e12:8.2f} {f(r)}  {f(rw)}')
print('drawn: CC', D['CC']['m'], 'units')

 CC units   CC/pF   PM model   PM model with wiring


       8     2.91    16.9      8.6


      12     4.36    34.3     25.9


      16     5.82    46.4     38.2


      20     7.27    55.5     47.6


      24     8.73    62.6     55.2


      32    11.63    73.3     66.4


      42    15.27    82.3     76.0
drawn: CC 20 units


### Branch current

`M5` and `M6` carry the branch. `M6` is at the current density of `M1` (50 nA per unit) and `M5` at that of `M4` (8.33 nA per unit), so the branch sets their unit counts and the vz pole through gm5. The table lists the branch for the counts that keep both densities.

In [18]:
print(' branch/nA  M6 units  M5 units  gm5/uS  vz pole/kHz (+wiring)  extra power/uW')
for ib in (25e-9, 50e-9, 100e-9, 200e-9):
    n6, n5 = ib / I_UNIT_N, ib / I_UNIT_P
    gm5 = per_um(lp, 'GM', LP, vsg_c, vsd_core) * Wp * n5
    cz = (N_VZ_GATES - D['M5']['m'] + n5) * cpu + n5 * cdd_pu
    print(f'{ib*1e9:9.0f} {n6:9.1f} {n5:9.1f} {gm5*1e6:7.3f} {f2pi(gm5/cz)/1e3:12.0f} ({f2pi(gm5/(cz+WIRE["vz"]))/1e3:4.0f}) {ib*VDD*1e6:14.3f}')
print(f"drawn: M5 {D['M5']['m']} units = {D['M5']['m']*I_UNIT_P*1e9:.0f} nA, M6 {D['M6']['m']} units = {D['M6']['m']*I_UNIT_N*1e9:.0f} nA")

 branch/nA  M6 units  M5 units  gm5/uS  vz pole/kHz (+wiring)  extra power/uW
       25       0.5       3.0   0.626           60 (  53)          0.037
       50       1.0       6.0   1.252          117 ( 104)          0.075
      100       2.0      12.0   2.503          224 ( 200)          0.150
      200       4.0      24.0   5.006          415 ( 373)          0.300
drawn: M5 12 units = 100 nA, M6 2 units = 100 nA


The vz pole moves in proportion to the branch current (224, 117 and 60 kHz at 100, 50 and 25 nA), and the README's old W/L 0.8 runs with 100, 50 and 25 nA gave -13, -17 and -42 deg of simulated phase margin, so the trend agrees. There is a second reason for 100 nA: at the 50 nA density of `M1`, a 50 nA branch is one `M6` unit and 25 nA half of one, and the house rule is at least 2 units per mirror element. 200 nA would double the pole again for 0.15 uW more.

The README also gives 0.15 uW per 100 nA of branch, the table agrees.

### Comparison with the drawing

In [19]:
rows = [
 ('M1 HV, 0.5u/10u',  I_CORE / I_UNIT_N, D['M1']['m'], 'core leg / 50 nA per unit'),
 ('M2 LV, 0.5u/10u',  I_CORE / I_UNIT_N, D['M2']['m'], 'K = 1, same total W/L as M1'),
 ('M6 HV, 0.5u/10u',  I_BR / I_UNIT_N, D['M6']['m'], 'branch at the M1 density'),
 ('M3 PMOS, 0.5u/5u', I_CORE / I_UNIT_P, D['M3']['m'], 'core leg / 8.33 nA'),
 ('M4 PMOS, 0.5u/5u', I_CORE / I_UNIT_P, D['M4']['m'], 'core leg / 8.33 nA'),
 ('M5 PMOS, 0.5u/5u', I_BR / I_UNIT_P, D['M5']['m'], 'branch / 8.33 nA'),
]
print(f"{'group':20s} {'derived':>8s} {'drawn':>6s}   basis")
for n, a, b, why in rows:
    print(f'{n:20s} {a:8.1f} {b:6d}   {why}' + ('' if abs(a - b) < 0.5 else '   <-- DISAGREES'))
print(f"{'R1 length / um':20s} {L_REQ:8.0f} {D['R1']['l']:6.0f}   model V_R / 200 nA, README sheet value")
print(f"{'V_R / mV':20s} {VR*1e3:8.0f} {'sim 421':>6s}")

group                 derived  drawn   basis
M1 HV, 0.5u/10u           4.0      4   core leg / 50 nA per unit
M2 LV, 0.5u/10u           4.0      4   K = 1, same total W/L as M1
M6 HV, 0.5u/10u           2.0      2   branch at the M1 density
M3 PMOS, 0.5u/5u         24.0     24   core leg / 8.33 nA
M4 PMOS, 0.5u/5u         24.0     24   core leg / 8.33 nA
M5 PMOS, 0.5u/5u         12.0     12   branch / 8.33 nA
R1 length / um            709    735   model V_R / 200 nA, README sheet value
V_R / mV                  419 sim 421


`MPD`, the vdd dummies of the core mirror, is `m=26` in the schematic. The README table said 27 and was corrected on 2026-11-06. It is a dummy and enters no number above.

Open points, as found: the README gm4 R of about 13 against 10.9 here, the README poles at `vy` (about 1 kHz against 0.14 kHz) and `vx` (about 150 kHz against 333 kHz) against the model. None of them changes a drawn size. The unit and the sim phase margin agree with the model within a few degrees.

The derivation reproduces the drawn unit counts (4, 4, 2, 24, 24, 12) from the 200 nA output, the 8.33 nA PMOS unit and the 50 nA NMOS unit. What stays open is the choice of the 50 nA unit current itself (the I/Is of about 4 in the README), which is a loop-dynamics choice the model above supports only through the vx pole and the phase margin sweep.